# Notebook 6 — Train, Tune, Evaluate

Baseline first so there's something concrete to beat, then a real model tuned
on val, then test gets touched exactly once at the end.

Reads: artifacts/train_features.parquet, val_features.parquet, test_features.parquet
Writes: artifacts/model.pkl, artifacts/results_summary.txt

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix,
)

ARTIFACTS_DIR = Path("artifacts")

train = pd.read_parquet(ARTIFACTS_DIR / "train_features.parquet")
val = pd.read_parquet(ARTIFACTS_DIR / "val_features.parquet")
test = pd.read_parquet(ARTIFACTS_DIR / "test_features.parquet")

drop_cols = ["order_id", "is_late"]
X_train, y_train = train.drop(columns=drop_cols), train["is_late"]
X_val, y_val = val.drop(columns=drop_cols), val["is_late"]
X_test, y_test = test.drop(columns=drop_cols), test["is_late"]

X_train.shape, X_val.shape, X_test.shape

((67533, 48), (14471, 48), (14472, 48))

## Baseline

Something dumb and fast, so any real model has an actual bar to clear. Given
how imbalanced this is (roughly 9% late in train), a model that just predicts
"on time" every single time would already look decent on accuracy alone,
which is exactly why accuracy isn't going to be the metric this notebook
tunes against.

In [2]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_val)
print("accuracy:", round(accuracy_score(y_val, baseline_preds), 4))
print("recall:  ", round(recall_score(y_val, baseline_preds), 4))
print("f1:      ", round(f1_score(y_val, baseline_preds), 4))

accuracy: 0.9466
recall:   0.0
f1:       0.0


There it is, close to 95% accuracy while catching zero late orders (recall
of 0). That's the trap with this kind of imbalance, a model can look great on
accuracy and still be completely useless. Anything trained here needs to
actually find late orders, not just default to the majority class.

## Picking the metric

Going with average precision (area under the precision-recall curve) as the
main number to optimize, it's a better fit than ROC-AUC when the positive
class is this rare, since it focuses on how well the model ranks and
identifies the minority class specifically, rather than getting inflated by
how easy it is to correctly call the huge majority of on-time orders. Still
tracking ROC-AUC and F1 alongside it, more for a sanity check than as the
thing being optimized.

## First real model

Random forest with class_weight balanced, this was the direction notebook 4
pointed toward: nothing on its own correlated strongly with is_late, which
usually means a model that can pick up interactions between features will
outperform something purely linear. Trying default-ish settings first before
tuning anything.

In [3]:
model = RandomForestClassifier(
    n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
val_preds = model.predict(X_val)

print("roc-auc:          ", round(roc_auc_score(y_val, val_probs), 4))
print("average precision:", round(average_precision_score(y_val, val_probs), 4))
print("f1 (0.5 threshold):", round(f1_score(y_val, val_preds), 4))

roc-auc:           0.7085
average precision: 0.1244
f1 (0.5 threshold): 0.0101


Already a real jump over the baseline's 0 recall, though nowhere near a
strong score on its own. Two more things worth trying before settling on
this: widening the hyperparameter search, and trying a gradient boosted
model alongside the random forest, since it's a genuinely different way of
building trees and sometimes does noticeably better on tabular data like this.

## Tuning the random forest, wider grid this time

Going beyond the first small grid, more combinations of tree count, depth,
leaf size, and this time also varying max_features, which controls how many
columns each split gets to consider, worth checking since there are a fair
number of one-hot columns now from the state buckets.

In [4]:
from sklearn.model_selection import ParameterGrid

rf_param_grid = list(ParameterGrid({
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5],
}))

print(f"{len(rf_param_grid)} combinations to try")

54 combinations to try


In [5]:
rf_results = []
for params in rf_param_grid:
    m = RandomForestClassifier(
        **params, class_weight="balanced", random_state=42, n_jobs=-1
    )
    m.fit(X_train, y_train)
    probs = m.predict_proba(X_val)[:, 1]
    ap = average_precision_score(y_val, probs)
    rf_results.append({**params, "val_avg_precision": ap})

rf_results_df = pd.DataFrame(rf_results).sort_values("val_avg_precision", ascending=False)
rf_results_df.head(10)

,max_depth,max_features,min_samples_leaf,n_estimators,val_avg_precision
44,20.0,sqrt,10,500,0.142203
6,NaN,sqrt,10,200,0.141781
7,NaN,sqrt,10,300,0.141357
42,20.0,sqrt,10,200,0.141254
43,20.0,sqrt,10,300,0.140509
5,NaN,sqrt,5,500,0.140140
19,10.0,sqrt,1,300,0.140127
8,NaN,sqrt,10,500,0.139793
4,NaN,sqrt,5,300,0.139742
18,10.0,sqrt,1,200,0.139718


In [6]:
best_rf_params = rf_results_df.iloc[0][["n_estimators", "max_depth", "min_samples_leaf", "max_features"]].to_dict()
best_rf_params["n_estimators"] = int(best_rf_params["n_estimators"])
best_rf_params["min_samples_leaf"] = int(best_rf_params["min_samples_leaf"])
best_rf_params["max_depth"] = None if pd.isna(best_rf_params["max_depth"]) else int(best_rf_params["max_depth"])

best_rf_avg_precision = rf_results_df.iloc[0]["val_avg_precision"]
print("best random forest params:", best_rf_params)
print("val average precision:", round(best_rf_avg_precision, 4))

best_rf = RandomForestClassifier(
    **best_rf_params, class_weight="balanced", random_state=42, n_jobs=-1
)
best_rf.fit(X_train, y_train)

best random forest params: {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 10, 'max_features': 'sqrt'}
val average precision: 0.1422


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fea

## Trying gradient boosting

Using HistGradientBoostingClassifier, it's built into sklearn so no extra
dependency, and handles this size of data quickly. It doesn't take a
class_weight argument the same way random forest does (depends on the
sklearn version), so computing sample weights manually instead and passing
those in at fit time, same effect: makes mistakes on the rare late class
count for more during training.

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)

gb_param_grid = list(ParameterGrid({
    "max_iter": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "learning_rate": [0.05, 0.1],
    "min_samples_leaf": [10, 20],
}))

print(f"{len(gb_param_grid)} combinations to try")

36 combinations to try


In [8]:
gb_results = []
for params in gb_param_grid:
    m = HistGradientBoostingClassifier(**params, random_state=42)
    m.fit(X_train, y_train, sample_weight=sample_weight_train)
    probs = m.predict_proba(X_val)[:, 1]
    ap = average_precision_score(y_val, probs)
    gb_results.append({**params, "val_avg_precision": ap})

gb_results_df = pd.DataFrame(gb_results).sort_values("val_avg_precision", ascending=False)
gb_results_df.head(10)

,learning_rate,max_depth,max_iter,min_samples_leaf,val_avg_precision
11,0.05,5.0,300,20,0.133507
9,0.05,5.0,200,20,0.133507
25,0.10,5.0,100,20,0.131845
27,0.10,5.0,200,20,0.131845
29,0.10,5.0,300,20,0.131845
26,0.10,5.0,200,10,0.131776
28,0.10,5.0,300,10,0.131776
24,0.10,5.0,100,10,0.131776
8,0.05,5.0,200,10,0.131703
10,0.05,5.0,300,10,0.131703


In [9]:
best_gb_params = gb_results_df.iloc[0][["max_iter", "max_depth", "learning_rate", "min_samples_leaf"]].to_dict()
best_gb_params["max_iter"] = int(best_gb_params["max_iter"])
best_gb_params["min_samples_leaf"] = int(best_gb_params["min_samples_leaf"])
best_gb_params["max_depth"] = None if pd.isna(best_gb_params["max_depth"]) else int(best_gb_params["max_depth"])

best_gb_avg_precision = gb_results_df.iloc[0]["val_avg_precision"]
print("best gradient boosting params:", best_gb_params)
print("val average precision:", round(best_gb_avg_precision, 4))

best_gb = HistGradientBoostingClassifier(**best_gb_params, random_state=42)
best_gb.fit(X_train, y_train, sample_weight=sample_weight_train)

best gradient boosting params: {'max_iter': 300, 'max_depth': 5, 'learning_rate': 0.05, 'min_samples_leaf': 20}
val average precision: 0.1335


,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",300
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing value

## Picking the winner

Whichever of the two tuned models scored higher on val average precision
gets carried forward, that's the one that gets a decision threshold picked
for it and gets evaluated on test.

In [10]:
if best_gb_avg_precision >= best_rf_avg_precision:
    model = best_gb
    model_name = "HistGradientBoostingClassifier"
    chosen_params = best_gb_params
else:
    model = best_rf
    model_name = "RandomForestClassifier"
    chosen_params = best_rf_params

print(f"going with {model_name}")
print(f"random forest val avg precision:      {best_rf_avg_precision:.4f}")
print(f"gradient boosting val avg precision:  {best_gb_avg_precision:.4f}")

going with RandomForestClassifier
random forest val avg precision:      0.1422
gradient boosting val avg precision:  0.1335


## Picking a decision threshold

Default 0.5 isn't necessarily the right cutoff for calling something "late"
when positives are this rare. Using the precision-recall curve on val (still
not touching test) to find the threshold that gives the best F1, that
becomes the cutoff used going forward instead of the default 0.5.

In [11]:
val_probs = model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

print(f"best threshold: {best_threshold:.3f}")
print(f"at that threshold on val -> precision: {precisions[best_idx]:.3f}, "
      f"recall: {recalls[best_idx]:.3f}, f1: {f1_scores[best_idx]:.3f}")

best threshold: 0.384
at that threshold on val -> precision: 0.152, recall: 0.360, f1: 0.214


## Test, once

Everything above (model choice, hyperparameters, threshold) got decided
using train and val only. This is the one and only time test gets touched,
using the exact model and threshold already locked in.

In [12]:
test_probs = model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

final_metrics = {
    "accuracy": accuracy_score(y_test, test_preds),
    "precision": precision_score(y_test, test_preds),
    "recall": recall_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds),
    "roc_auc": roc_auc_score(y_test, test_probs),
    "average_precision": average_precision_score(y_test, test_probs),
}
for k, v in final_metrics.items():
    print(f"{k:18s} {v:.4f}")

accuracy           0.7581
precision          0.1353
recall             0.4932
f1                 0.2124
roc_auc            0.7156
average_precision  0.1164


In [13]:
confusion_matrix(y_test, test_preds)

array([[10499,  3016],
       [  485,   472]])

Reading the confusion matrix: rows are actual (on time / late), columns are
predicted (on time / late). Worth looking at how many actual late orders got
caught versus missed, that's the recall number above, and how many on-time
orders got wrongly flagged as late, that's what's driving precision down if
it's lower than expected. Given the class imbalance, expecting precision to
be the harder number to get high, there just aren't many true late orders to
find relative to the pile of on-time ones the model has to sift through.

## Saving the model and a results summary

In [14]:
joblib.dump(model, ARTIFACTS_DIR / "model.pkl")
joblib.dump(best_threshold, ARTIFACTS_DIR / "decision_threshold.pkl")

summary = f"""Notebook 6 results summary

Baseline (predict majority class every time):
accuracy looked high but recall was 0, catches no late orders at all, not a
usable model, just a bar to beat.

Model comparison (val average precision):
random forest (tuned):      {best_rf_avg_precision:.4f}   params: {best_rf_params}
gradient boosting (tuned):  {best_gb_avg_precision:.4f}   params: {best_gb_params}
chosen model: {model_name}

Decision threshold (chosen by best F1 on val): {best_threshold:.3f}

Metric used for tuning: average precision (PR-AUC), chosen over accuracy
because of the roughly 9% positive rate, accuracy would reward a model for
just guessing the majority class.

Also added seller state, seller-to-customer distance, and product
weight/dimensions as features in this pass (notebook 1 and notebook 5), on
top of a wider hyperparameter search and trying gradient boosting alongside
random forest, following up on the first attempt's weak PR-AUC.

Final test set results (test touched once, after everything above was
already decided on train/val):
accuracy:  {final_metrics['accuracy']:.4f}
precision: {final_metrics['precision']:.4f}
recall:    {final_metrics['recall']:.4f}
f1:        {final_metrics['f1']:.4f}
roc-auc:   {final_metrics['roc_auc']:.4f}
avg precision: {final_metrics['average_precision']:.4f}
"""

with open(ARTIFACTS_DIR / "results_summary.txt", "w") as f:
    f.write(summary)

print(summary)

Notebook 6 results summary

Baseline (predict majority class every time):
accuracy looked high but recall was 0, catches no late orders at all, not a
usable model, just a bar to beat.

Model comparison (val average precision):
random forest (tuned):      0.1422   params: {'n_estimators': 500, 'max_depth': 20, 'min_samples_leaf': 10, 'max_features': 'sqrt'}
gradient boosting (tuned):  0.1335   params: {'max_iter': 300, 'max_depth': 5, 'learning_rate': 0.05, 'min_samples_leaf': 20}
chosen model: RandomForestClassifier

Decision threshold (chosen by best F1 on val): 0.384

Metric used for tuning: average precision (PR-AUC), chosen over accuracy
because of the roughly 9% positive rate, accuracy would reward a model for
just guessing the majority class.

Also added seller state, seller-to-customer distance, and product
weight/dimensions as features in this pass (notebook 1 and notebook 5), on
top of a wider hyperparameter search and trying gradient boosting alongside
random forest, followin

Started from a dummy baseline that made the imbalance problem obvious
(high accuracy, zero recall), picked average precision as the metric to
actually optimize instead of accuracy, then tuned both a random forest and a
gradient boosting model against val with a wider hyperparameter search than
the first pass, picked whichever scored higher, picked a decision threshold
off val's precision-recall curve, then evaluated on test exactly once with
everything already locked in.

This round also pulled in seller state, delivery distance, and product
weight/dimensions as features, which weren't in the table the first time
through, since the original feature set wasn't carrying much signal on its
own.

Saved the trained model, the chosen threshold, and a results summary under
artifacts/, that's the full chain from raw tables to a trained model, six
notebooks, one job each, everything reproducible from what got saved along
the way.